In [1]:
pip install pandas sqlalchemy pyodbc scikit-learn xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.8/69.5 MB 5.3 MB/s eta 0:00:14
    --------------------------------------- 1.6/69.5 MB 4.7 MB/s eta 0:00:15
   - -------------------------------------- 2.6/69.5 MB 4.4 MB/s eta 0:00:16
   - -------------------------------------- 3.1/69.5 MB 4.2 MB/s eta 0:00:16
   -- ------------------------------------- 3.7/69.5 MB 3.7 MB/s eta 0:00:18
   -- ------------------------------------- 3.9/69.5 MB 3.5 MB/s eta 0:00:19
   -- ------------------------------------- 3.9/69.5 MB 3.5 MB/s eta 0:00:19
   -- ------------------------------------- 4.2/69.5 MB 2.8 MB/s eta 0:00:24
   -- ------------------------------------- 4.5/69.5 MB 2.4 MB/s eta 0:00:28
   -- ------------------------------------- 4.5/69.5 MB 2.4 MB/s eta 0:00:28
   -- ------------------------------------- 4.7/69.5 MB 2.2 MB/s eta 0:00:30
   -- ------------------------------------- 5.0/69.5 MB 2.1 MB/s eta 0:00:31
   ---


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import pyodbc

# 1. Establish database connection (Update server name)
SERVER_NAME = "DESKTOP-V639HEA" 
DATABASE_NAME = "CreditRiskDB"
conn_str = f"mssql+pyodbc://{SERVER_NAME}/{DATABASE_NAME}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(conn_str)

# 2. Extract data via the SQL View
query = "SELECT * FROM v_Analytics_Credit_Risk;"
df = pd.read_sql(query, con=engine)

print(f"Data successfully extracted. Shape: {df.shape}")

C:\Users\DELL\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data successfully extracted. Shape: (32581, 15)


In [3]:
# Select features for the model
feature_cols = [
    'Person_Age', 'Person_Income', 'Person_Emp_Length', 'Loan_Amount', 
    'Loan_Interest_Rate', 'Loan_Percent_Income', 'Credit_History_Length_Years',
    'Home_Ownership', 'Loan_Intent', 'Loan_Grade', 'Historical_Default_File'
]

X = df[feature_cols]
y = df['Loan_Status_Default']

# One-Hot Encode categorical variables
X = pd.get_dummies(X, columns=['Home_Ownership', 'Loan_Intent', 'Loan_Grade', 'Historical_Default_File'], drop_first=True)

# Train/Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
    X, y, df['Loan_ID'], test_size=0.2, random_state=42, stratify=y
)

# Scale numerical columns for uniform distribution
scaler = StandardScaler()
numerical_cols = ['Person_Age', 'Person_Income', 'Person_Emp_Length', 'Loan_Amount', 'Loan_Interest_Rate', 'Loan_Percent_Income', 'Credit_History_Length_Years']

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

In [4]:
# Initialize and train XGBoost
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

# Predict probabilities for the ENTIRE dataset to push back to SQL
# We re-align the entire dataset features (scaled) for a complete database update
X_all = pd.get_dummies(df[feature_cols], columns=['Home_Ownership', 'Loan_Intent', 'Loan_Grade', 'Historical_Default_File'], drop_first=True)
X_all[numerical_cols] = scaler.transform(X_all[numerical_cols])

# Column 1 represents the probability of class 1 (Default)
df['Probability_of_Default'] = model.predict_proba(X_all)[:, 1]

# Segment the continuous probability into actionable risk tiers for the dashboard
df['Risk_Tier'] = pd.cut(
    df['Probability_of_Default'], 
    bins=[0, 0.15, 0.40, 0.70, 1.0], 
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical Risk']
)

print("Model training complete and risk scoring evaluated.")

Model training complete and risk scoring evaluated.


In [5]:
# Structure a clean dataframe for database insertion
df_predictions = df[['Loan_ID', 'Probability_of_Default', 'Risk_Tier']].copy()
df_predictions['Risk_Tier'] = df_predictions['Risk_Tier'].astype(str)

# Write to SQL Server
df_predictions.to_sql('Fact_Loan_Predictions', con=engine, if_exists='replace', index=False)

print("Predictions successfully written back to SQL Server as 'Fact_Loan_Predictions' table!")

Predictions successfully written back to SQL Server as 'Fact_Loan_Predictions' table!
